# Empirical Credit Risk and Spread Modeling

This notebook expands upon the theoretical foundations of structural models by introducing empirical, market-driven credit risk analysis. We bridge the gap between abstract probabilities and actionable fixed income metrics by focusing on the practical implementation of **Expected Loss (EL)**, **Credit Spreads** relative to default-free benchmarks, and the measurement of **Credit Spread Risk** using categorical rating scales from major NRSROs (Moody's, S&P, Fitch).

## 1. Expected Loss, Recovery & Rating Scales

While structural models derive default probabilities endogenously from asset volatility, empirical fixed income relies heavily on categorical credit ratings. The industry standard scales map directly to historical probabilities of default ($PD$):

* **Investment Grade (IG):**
    * **Moody's:** Aaa, Aa, A, Baa
    * **S&P / Fitch:** AAA, AA, A, BBB
* **High Yield (HY) / Speculative:**
    * **Moody's:** Ba, B, Caa, Ca, C
    * **S&P / Fitch:** BB, B, CCC, CC, C

The fundamental building block of credit risk is the **Expected Loss (EL)**. It is defined as a function of the Exposure at Default ($EAD$), the Probability of Default ($PD$), and the Loss Given Default ($LGD$):

$$EL = EAD \times PD \times LGD$$

The $LGD$ is strictly defined by the Recovery Rate ($RR$), historically anchored around 40% for senior unsecured corporate bonds:
$$LGD = 1 - RR$$

## 2. Credit Spreads and Benchmark Pricing

The compensation an investor demands for bearing the Expected Loss (plus a liquidity and risk premium) is the **Credit Spread** ($s$). For a non-Treasury corporate bond, the yield to maturity ($y$) is priced relative to a default-free benchmark (typically the sovereign yield curve, $y_{rf}$):

$$y = y_{rf} + s$$

In a continuous-time, risk-neutral framework assuming a constant hazard rate $\lambda$ and instantaneous recovery $R$, the spread can be closely approximated by the expected loss rate:
$$s \approx \lambda (1 - R)$$

## 3. Modeling Credit Spread Risk

Credit Spread Risk is the danger that a bond's price will decline if the market demands higher compensation for risk, causing spreads to "widen". We quantify this sensitivity using **Spread Duration** (often equal to Modified Duration, $D_{mod}$, for fixed-rate bonds).

For a given parallel shift in the credit spread ($\Delta s$), the approximate percentage change in the bond's price ($P$) is evaluated as:

$$\frac{\Delta P}{P} \approx -D_{mod} \times \Delta s$$

Below, we implement an interactive model connecting these rating categories, expected losses, and spread durations.

In [ ]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# 1. Define the Rating Scales and Baseline Market Data
# Mapping typical S&P/Fitch ratings to 5-yr cumulative PDs (%) and baseline Spread levels (bps)
rating_data = {
    'AAA': {'PD': 0.05, 'Spread_bps': 54},
    'AA':  {'PD': 0.10, 'Spread_bps': 65},
    'A':   {'PD': 0.20, 'Spread_bps': 85},
    'BBB': {'PD': 0.50, 'Spread_bps': 120},
    'BB':  {'PD': 2.00, 'Spread_bps': 250},
    'B':   {'PD': 5.00, 'Spread_bps': 400},
    'CCC': {'PD': 15.00, 'Spread_bps': 900}
}

def analyze_credit_risk(rating, benchmark_yield, recovery_rate, duration, spread_shock):
    """
    Calculates and displays expected loss, total yield, and spread risk 
    based on categorical rating inputs and market parameters.
    """
    # Extract base metrics based on the selected rating
    pd_pct = rating_data[rating]['PD']
    base_spread = rating_data[rating]['Spread_bps']
    
    # Expected Loss Calculation (assuming $100 face value)
    lgd_pct = 100 - recovery_rate
    expected_loss_per_100 = 100 * (pd_pct / 100.0) * (lgd_pct / 100.0)
    
    # Yield Dynamics
    total_yield = benchmark_yield + (base_spread / 100.0) 
    
    # Spread Risk Modeling (Price Impact)
    # dP/P ~ -Duration * dSpread
    price_impact_pct = -duration * (spread_shock / 10000.0) * 100.0
    
    # Formatted Output Display
    print(f"\033[1m--- Credit Profile: {rating} ---\033[0m")
    print(f"Probability of Default (PD): {pd_pct:5.2f}%")
    print(f"Loss Given Default (LGD):    {lgd_pct:5.2f}%")
    print(f"Expected Loss (per $100):    ${expected_loss_per_100:.4f}")
    print("-" * 35)
    print(f"Benchmark Yield:             {benchmark_yield:5.2f}%")
    print(f"Credit Spread:               {base_spread:4d} bps")
    print(f"\033[1mTotal Bond Yield:            {total_yield:5.2f}%\033[0m")
    print("-" * 35)
    print(f"Spread Shock (Widening):     +{spread_shock} bps")
    print(f"\033[91mEstimated Price Impact:      {price_impact_pct:.2f}%\033[0m")

# 2. Deploy Interactive Jupyter Widget
widgets.interact(
    analyze_credit_risk,
    rating=widgets.Dropdown(options=list(rating_data.keys()), value='BBB', description='Rating:'),
    benchmark_yield=widgets.FloatSlider(value=4.25, min=0.0, max=10.0, step=0.25, description='BM Yield (%):'),
    recovery_rate=widgets.FloatSlider(value=40.0, min=0.0, max=100.0, step=5.0, description='Recovery (%):'),
    duration=widgets.FloatSlider(value=5.0, min=0.5, max=30.0, step=0.5, description='Duration (yr):'),
    spread_shock=widgets.IntSlider(value=50, min=0, max=500, step=10, description='Shock (bps):')
);

interactive(children=(Dropdown(description='Rating:', index=3, options=('AAA', 'AA', 'A', 'BBB', 'BB', 'B', 'C…